In [6]:
import json
from datasets import load_dataset

In [7]:
datasets = load_dataset('hate-speech-portuguese/hate_speech_portuguese', split='train[:10%]')


In [8]:
print(datasets)

Dataset({
    features: ['text', 'label', 'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'],
    num_rows: 567
})


In [9]:
datasets = datasets.remove_columns([
    'hatespeech_G1', 'annotator_G1', 'hatespeech_G2', 'annotator_G2', 'hatespeech_G3', 'annotator_G3'
])

In [10]:
print(datasets)

Dataset({
    features: ['text', 'label'],
    num_rows: 567
})


In [11]:
datasets = datasets.train_test_split(test_size=0.2)

In [12]:
print(datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 453
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 114
    })
})


In [13]:
datasets["train"]["text"]

['A música não tem cores, nem classes é para todos.\nUm simples projeto que muda a vida de muitas pessoas. \nhttps://t.co/gVFuwtXodn',
 'A forma como você se vê não altera o que você é de fato! https://t.co/F1vbtmfKGs',
 'Agenda Março . Ideologia de Gênero. https://t.co/vhpKZjqQJD',
 'Ah, porque o nacionalismo deste gajo é económico e é tão económico quanto o do PCP. Com a diferença de o PCP não querer Angola de volta.',
 'A mulher é posdoctor nas fisica, pesquisadora e ainda arraza na cozinha uma mulher dessas bicho   #MasterChefBR',
 'A Esquerda é o próprio Zumbi dos Palmares:\nQuer libertar os negros de uma escravidão pra escraviza-los em outra https://t.co/c1XOaNIPmI',
 'A Justiça tem que estar ao serviço de todos e não à mercê de privilégios de uns quantos.\n@govpt @antoniocostapm \nhttps://t.co/s4IjrxP8vM',
 "'A maior ambição da mulher é despertar o amor.' #MulherDeVerdade",
 '@ajulysantos e era de direita.... essa menina tem transtorno de personalidade ou algo pior...',
 '@Ademi

In [14]:
def removeN(example):
    example['text'] = example['text'].replace("\n", " ")
    return example

In [15]:
datasets = datasets.map(removeN)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [16]:
datasets['train'][3]

{'text': 'Ah, porque o nacionalismo deste gajo é económico e é tão económico quanto o do PCP. Com a diferença de o PCP não querer Angola de volta.',
 'label': 0}

In [17]:
# label 0 -> No hate speech
# label 1 -> Hate speech

def labelChange(example):
    example["label_text"] = 'No Hate Speech' if example['label'] == 0 else 'Hate Speech'
    return example


In [18]:
datasets = datasets.map(labelChange)

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Map:   0%|          | 0/114 [00:00<?, ? examples/s]

In [19]:
datasets = datasets.remove_columns(['label'])

In [20]:
print(datasets['train'][0])

{'text': 'A música não tem cores, nem classes é para todos. Um simples projeto que muda a vida de muitas pessoas.  https://t.co/gVFuwtXodn', 'label_text': 'No Hate Speech'}


In [ ]:
# CONSTRUCAO DO OBJETO PARA OPENAI

def dataset_to_jsonl(dataset, file_name):
    with open(file_name, 'w', encoding='utf-8') as f:
        for example in dataset:
            json_obj = {"messages": [
                {"role": "system", "content": "Seu trabalho é classificar os comentários do usuário em Hate Speech e No Hate Speech."},
                {"role": "user", "content": example['text']},
                {"role": "assistant", "content": example['label_text']},
            ]}
            f.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

In [31]:
dataset_to_jsonl(datasets['train'], 'train.jsonl')

In [32]:
dataset_to_jsonl(datasets['test'], 'validation.jsonl')

In [34]:
from openai import OpenAI
import os

In [ ]:
os.environ["OPENAI_API_KEY"] = ""

In [36]:
client = OpenAI()

In [37]:
client.files.create(
    file=open("train.jsonl", 'rb'),
    purpose="fine-tune"
)

FileObject(id='file-5Z8ugesCePd1b8WV9pDNC7', bytes=146246, created_at=1778343311, filename='train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [38]:
client.files.create(
    file=open("validation.jsonl", 'rb'),
    purpose="fine-tune"
)

FileObject(id='file-BoA2Thq2W73rpuyvD1bWtP', bytes=37576, created_at=1778343321, filename='validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [ ]:
# Deprecated

client.fine_tuning.jobs.create(
    training_file='file-5Z8ugesCePd1b8WV9pDNC7',
    validation_file='file-BoA2Thq2W73rpuyvD1bWtP',
    model='gpt-3.5-turbo'
)

In [21]:
# CONSTRUCAO DO OBJETO PARA AWS BEDROCK

def dataset_to_jsonlAWS(dataset, file_name):
    with open(file_name, 'w', encoding='utf-8') as f:
        for example in dataset:
            json_obj = {
                "prompt": example['text'],
                "completion": example['label_text']
            }
            f.write(json.dumps(json_obj, ensure_ascii=False) + '\n')

In [22]:
dataset_to_jsonlAWS(datasets['train'], 'train.jsonl')

In [23]:

dataset_to_jsonlAWS(datasets['test'], 'validation.jsonl')